In [1]:
from instance import Instance, compute_or_load_metrics
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import numpy as np

In [2]:
instance_dir = "/home/pedroldm/MSc/cbm/instances"
instance_names = []
num_rows = []
num_cols = []
total_ones = []
empty_rows = []
empty_cols = []
row_subsets = []
col_subsets = []

In [3]:
metrics = compute_or_load_metrics(
    instance_dir=instance_dir,
    cache_file="instance_metrics.json",
    max_workers=12
)

Loading cached metrics from instance_metrics.json


In [4]:
%matplotlib widget

def plot_instance_dashboard(instances):
    """
    Beautiful interactive Plotly dashboard for CBM instance analysis.

    Features
    --------
    - Dropdown to switch between instances
    - Professional Plotly styling
    - Hover tooltips
    - Zoom/pan
    - Responsive layout

    Parameters
    ----------
    instances : list[dict]
        List of instance analysis dictionaries.

    Returns
    -------
    plotly.graph_objects.Figure
    """

    # Create subplot layout
    fig = make_subplots(
        rows=3,
        cols=2,
        row_heights=[0.28, 0.32, 0.40],
        column_widths=[0.5, 0.5],
        specs=[
            [{"type": "indicator"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "scatter"}],
            [{"type": "table", "colspan": 2}, None],
        ],
        subplot_titles=(
            "Matrix Density",
            "Singletons",
            "Subset Counts",
            "Singleton Index Distribution",
            "Detailed Metrics",
        ),
        vertical_spacing=0.08,
    )

    traces_per_instance = 6

    for instance_idx, d in enumerate(instances):
        visible = instance_idx == 0

        density = 100 * d["total_ones"] / (d["num_rows"] * d["num_cols"])

        # 1. Density gauge
        fig.add_trace(
            go.Indicator(
                mode="gauge+number",
                value=density,
                number={"suffix": "%", "valueformat": ".2f"},
                gauge={
                    "axis": {"range": [0, max(5, density * 1.3)]},
                    "bar": {"thickness": 0.3},
                },
                title={"text": "Density"},
                visible=visible,
            ),
            row=1,
            col=1,
        )

        # 2. Singleton comparison
        fig.add_trace(
            go.Bar(
                x=["Singleton Rows", "Singleton Columns"],
                y=[d["singleton_rows"], d["singleton_cols"]],
                text=[d["singleton_rows"], d["singleton_cols"]],
                textposition="outside",
                visible=visible,
                hovertemplate="%{x}: %{y}<extra></extra>",
            ),
            row=1,
            col=2,
        )

        # 3. Subset counts
        fig.add_trace(
            go.Bar(
                x=["Row Subsets", "Column Subsets"],
                y=[d["row_subsets"], d["col_subsets"]],
                text=[d["row_subsets"], d["col_subsets"]],
                textposition="outside",
                visible=visible,
                hovertemplate="%{x}: %{y}<extra></extra>",
            ),
            row=2,
            col=1,
        )

        # 4. Singleton row indices
        fig.add_trace(
            go.Scatter(
                x=d["singleton_row_indices"],
                y=["Rows"] * len(d["singleton_row_indices"]),
                mode="markers",
                name="Rows",
                marker={"size": 12, "symbol": "circle"},
                visible=visible,
                hovertemplate="Row %{x}<extra></extra>",
                showlegend=False,
            ),
            row=2,
            col=2,
        )

        # 5. Singleton column indices
        fig.add_trace(
            go.Scatter(
                x=d["singleton_col_indices"],
                y=["Columns"] * len(d["singleton_col_indices"]),
                mode="markers",
                name="Columns",
                marker={"size": 8, "symbol": "diamond"},
                visible=visible,
                hovertemplate="Column %{x}<extra></extra>",
                showlegend=False,
            ),
            row=2,
            col=2,
        )

        # 6. Metrics table
        fig.add_trace(
            go.Table(
                header=dict(
                    values=["Metric", "Value"],
                    align="left",
                ),
                cells=dict(
                    values=[
                        [
                            "Filename",
                            "Rows",
                            "Columns",
                            "Total Ones",
                            "Density",
                            "Empty Rows",
                            "Empty Columns",
                            "Singleton Rows",
                            "Singleton Columns",
                            "Row Subsets",
                            "Column Subsets",
                        ],
                        [
                            d["filename"],
                            d["num_rows"],
                            d["num_cols"],
                            d["total_ones"],
                            f"{density:.3f}%",
                            d["empty_rows"],
                            d["empty_cols"],
                            d["singleton_rows"],
                            d["singleton_cols"],
                            d["row_subsets"],
                            d["col_subsets"],
                        ],
                    ],
                    align="left",
                ),
                visible=visible,
            ),
            row=3,
            col=1,
        )

    # Dropdown buttons
    buttons = []

    for i, d in enumerate(instances):
        visible = [False] * len(fig.data)

        start = i * traces_per_instance
        end = start + traces_per_instance

        for j in range(start, end):
            visible[j] = True

        buttons.append(
            dict(
                label=d["filename"],
                method="update",
                args=[
                    {"visible": visible},
                    {
                        "title": {
                            "text": f"CBM Instance Analysis — {d['filename']}"
                        }
                    },
                ],
            )
        )

    # Layout
    fig.update_layout(
        template="plotly_white",
        height=1000,
        title={
            "text": f"CBM Instance Analysis — {instances[0]['filename']}",
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 28},
        },
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=0.5,
                xanchor="center",
                y=1.15,
                yanchor="top",
                showactive=True,
            )
        ],
        margin=dict(t=140, l=40, r=40, b=40),
    )

    # Better axis labels
    fig.update_xaxes(title_text="Index", row=2, col=2)
    fig.update_yaxes(title_text="", row=2, col=2)

    return fig


In [5]:
with open("instance_metrics.json", "r") as f:
    instance_metrics = json.load(f)
    fig = plot_instance_dashboard(instance_metrics)
    fig.show()

## 1. Critérios Baseados na Dispersão e Perfil dos Blocos

Em vez de olhar apenas para o total de transições, você pode analisar a "qualidade" ou o formato dos blocos gerados.

### Maior Tamanho do Maior Bloco (Max Block Size)
* **Como funciona:** Conte o comprimento de todos os blocos contíguos de $1$s na matriz. Entre duas soluções com o mesmo custo, prefira aquela que possui o maior bloco isolado de $1$s.
* **Por que ajuda:** Blocos longos de $1$s tendem a ser estruturas estáveis e ótimas. É mais fácil o algoritmo ajustar as "beiradas" de colunas fragmentadas se as colunas que formam o grande bloco central já estiverem rigidamente unidas.

### Menor Desvio Padrão do Tamanho dos Blocos
* **Como funciona:** Calcule a variância ou o desvio padrão dos comprimentos de todos os blocos de $1$s. Prefira a solução onde os tamanhos dos blocos são mais homogêneos (ou mais heterogêneos, dependendo da instância, mas geralmente a homogeneidade indica maior alinhamento global).

---

## 2. Critérios Baseados na Distância entre Colunas (Matriz de Afinidade)

Como você já calcula uma `diffMatrix` (matriz de distância/dissimilaridade entre colunas) para rodar o LKH, você pode usar essa mesma informação para o desempate.

### Menor Distância Total Absoluta (Distância Euclidiana/Hamming Adjacente)
* **Como funciona:** A função objetivo padrão do CBM conta as transições $0 \rightarrow 1$. No entanto, você pode somar a distância real (frequência com que as colunas diferem) apenas entre colunas vizinhas na solução.
* **Por que ajuda:** Duas soluções podem ter o mesmo número de blocos, mas uma delas pode ter colunas vizinhas que são quase idênticas (diferem por apenas 1 ou 2 bits), enquanto a outra tem vizinhos muito díspares que por "coincidência" mantiveram o mesmo número de blocos. A solução com menor distância total adjacente está mais "compacta" e propensa a melhorar.

---

## 3. Critérios Baseados no Potencial de Movimentação (Vizinhança)

### Contagem de "Quase-Melhorias" (Look-ahead de 1-Move)
* **Como funciona:** Avalie quantos movimentos na vizinhança da solução (como um `REINSERTION` ou `SWAP`) resultariam em um custo imediatamente igual ou ligeiramente pior. Prefira a solução que maximiza o número de vizinhos com custo igual ou menor.
* **Por que ajuda:** Isso mede a conectividade do platô. Uma solução que está cercada de vizinhos que mantêm o mesmo custo oferece mais caminhos para o algoritmo continuar explorando sem ficar preso em um ótimo local estrito (vazio de alternativas).

---

## 4. Critérios Baseados na Concentração de Zeros ("Buracos")

### Minimização do "Espaço Vazio" Interno (Gap Cost)
* **Como funciona:** Para cada linha, encontre o primeiro $1$ e o último $1$. Conte quantos $0$s existem entre eles (os "gaps" ou buracos). Escolha a solução que minimiza o total de zeros internos.
* **Por que ajuda:** O CBM clássico pune a abertura de novos blocos. Se uma solução tem menos zeros intermediários, significa que os $1$s estão sendo empurrados para o centro da atividade da linha, reduzindo o espalhamento, o que facilita o agrupamento completo em passos subsequentes da metaheurística.